# Financial Data Extraction & Initial Analysis
**BCG GenAI Job Simulation, Task 1 | Client: Global Finance Corp. (GFC)**

## Objective
Extract five key financial metrics from the FY2023–FY2025 10-K filings of Microsoft, Apple, and Tesla, then analyze year-over-year trends with pandas. The results inform which metrics and insights an AI-powered financial chatbot should be able to report.

## Data source
| Company | SEC CIK | Fiscal year ends | Filings used |
|---|---|---|---|
| Microsoft | 789019 | June 30 | 10-K for FY2023, FY2024, FY2025 |
| Apple | 320193 | Last Saturday of September | 10-K for FY2023, FY2024, FY2025 |
| Tesla | 1318605 | December 31 | 10-K for FY2023, FY2024, FY2025 |

## Methodology
1. Located each 10-K on SEC EDGAR and navigated to **Item 8 (Financial Statements and Supplementary Data)**, which contains the audited statements.
2. Manually extracted Total Revenue, Net Income, Total Assets, Total Liabilities, and Cash Flow from Operating Activities.
3. Recorded all figures in **USD millions**, as presented in the filings.
4. For Tesla, used *Net income attributable to common stockholders* for consistency with the other two companies.
5. Compiled the data in Excel, exported it to CSV, and loaded it into pandas for cleaning, validation, and analysis.
6. Cross-checked all 45 extracted values against the SEC's XBRL company facts data. All values matched.

## 1. Setup

In [1]:
import pandas as pd

# Show all columns and format numbers with thousands separators
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Load the data
The CSV has one row per company per fiscal year (3 companies × 3 years = 9 rows). All dollar figures are in USD millions.

In [2]:
df = pd.read_csv("financial_data.csv")

# Strip stray whitespace left over from the spreadsheet export
df.columns = df.columns.str.strip()
df["Company"] = df["Company"].astype(str).str.strip()

df

,Company,Fiscal Year,Total Revenue,Net Income,Total Assets,Total Liabilities,Cash Flow from Operating Activities
0,Microsoft,2023,"211,915","72,361","411,976","205,753","87,582"
1,Microsoft,2024,"245,122","88,136","512,163","243,686","118,548"
2,Microsoft,2025,"281,724","101,832","619,003","275,524","136,162"
3,Tesla,2023,"96,773","14,997","106,618","43,009","13,256"
4,Tesla,2024,"97,690","7,091","122,070","48,390","14,923"
5,Tesla,2025,"94,827","3,794","137,806","54,941","14,747"
6,Apple,2023,"383,285","96,995","352,583","290,437","110,543"
7,Apple,2024,"391,035","93,736","364,980","308,030","118,254"
8,Apple,2025,"416,161","112,010","359,241","285,508","111,482"


## 3. Clean and validate
Before any analysis, the figures are converted to true numbers and checked for blanks, duplicates, scale errors, and swapped columns.

In [3]:
METRICS = [
    "Total Revenue",
    "Net Income",
    "Total Assets",
    "Total Liabilities",
    "Cash Flow from Operating Activities",
]

# Convert metrics to numbers, handling $ signs, commas, and (negative) values.
# Anything unparseable becomes NaN so the validation step catches it.
for col in METRICS:
    cleaned = (
        df[col].astype(str).str.strip()
        .str.replace(r"^\((.*)\)$", r"-\1", regex=True)
        .str.replace(r"[$,]", "", regex=True)
    )
    df[col] = pd.to_numeric(cleaned, errors="coerce")

df["Fiscal Year"] = pd.to_numeric(df["Fiscal Year"], errors="coerce").astype("Int64")
df.dtypes

Company                                  str
Fiscal Year                            Int64
Total Revenue                          int64
Net Income                             int64
Total Assets                           int64
Total Liabilities                      int64
Cash Flow from Operating Activities    int64
dtype: object

In [4]:
# Check 1: missing or unparseable values
print("Missing values per column:")
print(df[METRICS + ["Fiscal Year"]].isna().sum().to_string(), "\n")

# Check 2: one row per company per year
print("Rows per company:")
print(df["Company"].value_counts().to_string())
dupes = df.duplicated(subset=["Company", "Fiscal Year"], keep=False).sum()
print(f"Duplicate company-year rows: {dupes}\n")

# Check 3: all three companies have positive equity, so liabilities should be below assets
bad_balance = df[df["Total Liabilities"] >= df["Total Assets"]]
print(f"Rows where liabilities >= assets: {len(bad_balance)}\n")

# Check 4: revenue recorded in USD millions should be 5-6 digits
digits = (
    df.set_index(["Company", "Fiscal Year"])["Total Revenue"]
      .dropna().abs().round().astype("int64").astype(str).str.len()
)
print("Revenue digit count:")
print(digits.to_string())

Missing values per column:
Total Revenue                          0
Net Income                             0
Total Assets                           0
Total Liabilities                      0
Cash Flow from Operating Activities    0
Fiscal Year                            0 

Rows per company:
Company
Microsoft    3
Tesla        3
Apple        3
Duplicate company-year rows: 0

Rows where liabilities >= assets: 0

Revenue digit count:
Company    Fiscal Year
Microsoft  2023           6
           2024           6
           2025           6
Tesla      2023           5
           2024           5
           2025           5
Apple      2023           6
           2024           6
           2025           6


## 4. Year-over-year growth
Percentage change from each fiscal year to the next, calculated separately for each company. FY2023 has no prior year in this dataset, so its growth values are blank (NaN) by design.

In [5]:
# Sort chronologically within each company so pct_change compares each year to the prior year
df = df.sort_values(["Company", "Fiscal Year"]).reset_index(drop=True)

df["Revenue Growth (%)"] = df.groupby(["Company"])["Total Revenue"].pct_change() * 100
df["Net Income Growth (%)"] = df.groupby(["Company"])["Net Income"].pct_change() * 100
df["Total Assets Growth (%)"] = df.groupby(["Company"])["Total Assets"].pct_change() * 100
df["Total Liabilities Growth (%)"] = df.groupby(["Company"])["Total Liabilities"].pct_change() * 100
df["Operating Cash Flow Growth (%)"] = df.groupby(["Company"])["Cash Flow from Operating Activities"].pct_change() * 100

# FY2023 has no prior year in the dataset, so its growth values stay NaN
growth_cols = [c for c in df.columns if c.endswith("Growth (%)")]
df[["Company", "Fiscal Year"] + growth_cols]

,Company,Fiscal Year,Revenue Growth (%),Net Income Growth (%),Total Assets Growth (%),Total Liabilities Growth (%),Operating Cash Flow Growth (%)
0,Apple,2023,NaN,NaN,NaN,NaN,NaN
1,Apple,2024,2.02,-3.36,3.52,6.06,6.98
2,Apple,2025,6.43,19.50,-1.57,-7.31,-5.73
3,Microsoft,2023,NaN,NaN,NaN,NaN,NaN
4,Microsoft,2024,15.67,21.80,24.32,18.44,35.36
5,Microsoft,2025,14.93,15.54,20.86,13.07,14.86
6,Tesla,2023,NaN,NaN,NaN,NaN,NaN
7,Tesla,2024,0.95,-52.72,14.49,12.51,12.58
8,Tesla,2025,-2.93,-46.50,12.89,13.54,-1.18


## 5. Financial ratios
Raw dollar figures favor the largest company. Ratios put all three on equal footing and are more useful for comparison.

| Ratio | Formula | Interpretation |
|---|---|---|
| Net margin | Net income ÷ Revenue | Profit kept from each dollar of sales |
| Debt ratio | Total liabilities ÷ Total assets | Share of assets funded by obligations (higher = more leverage) |
| Cash conversion | Operating cash flow ÷ Net income | Above 1.0 means profits are backed by real cash |
| OCF margin | Operating cash flow ÷ Revenue | Cash generated from each dollar of sales |

In [6]:
df["Net Margin (%)"] = df["Net Income"] / df["Total Revenue"] * 100
df["Debt Ratio (%)"] = df["Total Liabilities"] / df["Total Assets"] * 100
df["Cash Conversion (x)"] = df["Cash Flow from Operating Activities"] / df["Net Income"]
df["OCF Margin (%)"] = df["Cash Flow from Operating Activities"] / df["Total Revenue"] * 100

# Assets minus liabilities. Slightly overstates Tesla's equity, since Tesla also
# reports a small redeemable noncontrolling interest between liabilities and equity.
df["Implied Equity"] = df["Total Assets"] - df["Total Liabilities"]

ratio_cols = ["Net Margin (%)", "Debt Ratio (%)", "Implied Equity", "Cash Conversion (x)", "OCF Margin (%)"]
df[["Company", "Fiscal Year"] + ratio_cols]

,Company,Fiscal Year,Net Margin (%),Debt Ratio (%),Implied Equity,Cash Conversion (x),OCF Margin (%)
0,Apple,2023,25.31,82.37,62146,1.14,28.84
1,Apple,2024,23.97,84.40,56950,1.26,30.24
2,Apple,2025,26.92,79.48,73733,1.00,26.79
3,Microsoft,2023,34.15,49.94,206223,1.21,41.33
4,Microsoft,2024,35.96,47.58,268477,1.35,48.36
5,Microsoft,2025,36.15,44.51,343479,1.34,48.33
6,Tesla,2023,15.50,40.34,63609,0.88,13.70
7,Tesla,2024,7.26,39.64,73680,2.10,15.28
8,Tesla,2025,4.00,39.87,82865,3.89,15.55


## 6. Aggregate views
The same data grouped three ways: two-year compound growth by company, side-by-side comparisons by year, and the leader on each metric each year.

In [7]:
# CAGR = (end / start) ^ (1 / years) - 1
# Only valid when both values are positive, which holds for every metric here
first_row = df.groupby("Company").first()
last_row = df.groupby("Company").last()
n_years = (last_row["Fiscal Year"] - first_row["Fiscal Year"]).astype(float)

cagr = pd.DataFrame({
    f"{col} CAGR (%)": ((last_row[col] / first_row[col]) ** (1 / n_years) - 1) * 100
    for col in METRICS
})
cagr

,Total Revenue CAGR (%),Net Income CAGR (%),Total Assets CAGR (%),Total Liabilities CAGR (%),Cash Flow from Operating Activities CAGR (%)
Company,,,,,
Apple,4.20,7.46,0.94,-0.85,0.42
Microsoft,15.30,18.63,22.58,15.72,24.69
Tesla,-1.01,-49.70,13.69,13.02,5.47


In [8]:
compare_cols = ["Total Revenue", "Net Income", "Net Margin (%)", "Cash Flow from Operating Activities", "Debt Ratio (%)"]

for col in compare_cols:
    print(f"\n{col}")
    display(df.pivot_table(index="Company", columns="Fiscal Year", values=col))


Total Revenue


Fiscal Year,2023,2024,2025
Company,,,
Apple,"383,285.00","391,035.00","416,161.00"
Microsoft,"211,915.00","245,122.00","281,724.00"
Tesla,"96,773.00","97,690.00","94,827.00"



Net Income


Fiscal Year,2023,2024,2025
Company,,,
Apple,"96,995.00","93,736.00","112,010.00"
Microsoft,"72,361.00","88,136.00","101,832.00"
Tesla,"14,997.00","7,091.00","3,794.00"



Net Margin (%)


Fiscal Year,2023,2024,2025
Company,,,
Apple,25.31,23.97,26.92
Microsoft,34.15,35.96,36.15
Tesla,15.50,7.26,4.00



Cash Flow from Operating Activities


Fiscal Year,2023,2024,2025
Company,,,
Apple,"110,543.00","118,254.00","111,482.00"
Microsoft,"87,582.00","118,548.00","136,162.00"
Tesla,"13,256.00","14,923.00","14,747.00"



Debt Ratio (%)


Fiscal Year,2023,2024,2025
Company,,,
Apple,82.37,84.40,79.48
Microsoft,49.94,47.58,44.51
Tesla,40.34,39.64,39.87


In [9]:
# Company with the highest value for each metric in each year
leader_cols = METRICS + ["Net Margin (%)", "OCF Margin (%)"]
leaders = df.set_index("Company").groupby("Fiscal Year")[leader_cols].idxmax()
leaders

,Total Revenue,Net Income,Total Assets,Total Liabilities,Cash Flow from Operating Activities,Net Margin (%),OCF Margin (%)
Fiscal Year,,,,,,,
2023,Apple,Apple,Microsoft,Apple,Apple,Microsoft,Microsoft
2024,Apple,Apple,Microsoft,Apple,Microsoft,Microsoft,Microsoft
2025,Apple,Apple,Microsoft,Apple,Microsoft,Microsoft,Microsoft


## 7. Summary statements
One sentence per company for the most recent fiscal year, built from the precomputed metrics. This is how the chatbot can return figures without calculating them at query time.

In [10]:
latest = df.sort_values("Fiscal Year").groupby("Company").tail(1)

def direction(value):
    return "up" if value >= 0 else "down"

for _, row in latest.iterrows():
    rev_g = row["Revenue Growth (%)"]
    ni_g = row["Net Income Growth (%)"]
    print(
        f"{row['Company']} FY{row['Fiscal Year']}: "
        f"revenue ${row['Total Revenue']:,.0f}M ({direction(rev_g)} {abs(rev_g):.1f}% YoY), "
        f"net income ${row['Net Income']:,.0f}M ({direction(ni_g)} {abs(ni_g):.1f}% YoY), "
        f"net margin {row['Net Margin (%)']:.1f}%, "
        f"operating cash flow ${row['Cash Flow from Operating Activities']:,.0f}M."
    )

Apple FY2025: revenue $416,161M (up 6.4% YoY), net income $112,010M (up 19.5% YoY), net margin 26.9%, operating cash flow $111,482M.
Microsoft FY2025: revenue $281,724M (up 14.9% YoY), net income $101,832M (up 15.5% YoY), net margin 36.1%, operating cash flow $136,162M.
Tesla FY2025: revenue $94,827M (down 2.9% YoY), net income $3,794M (down 46.5% YoY), net margin 4.0%, operating cash flow $14,747M.


## 8. Findings

### Revenue
- Microsoft grew revenue fastest, at a 15.3% compound annual rate from FY2023 to FY2025 (211.9B to 281.7B), compared with 4.2% for Apple and -1.0% for Tesla.
- Apple's revenue growth accelerated from 2.0% in FY2024 to 6.4% in FY2025. The FY2024 figure is understated because FY2023 included an extra 53rd week.
- Tesla was the only company whose revenue shrank, falling 2.9% to 94.8B in FY2025. Automotive revenue fell 9.8% (77.1B to 69.5B) on fewer deliveries (down 8.6%), lower average selling prices, and lower regulatory credit revenue. Growth in energy storage (+26.6%) and services (+18.9%) offset about 62% of that decline.

### Profitability
- Microsoft has the highest net margin, and it is still widening (34.1%, 36.0%, 36.2%). Net income outgrew revenue in both years (+21.8% vs. +15.7% in FY2024; +15.5% vs. +14.9% in FY2025), so costs rose more slowly than sales.
- Apple's revenue rose 2.0% in FY2024 while net income fell 3.4%, due to a one-time tax charge of about 10.2B from a European Commission ruling on its Irish taxes. With that charge gone, FY2025 net income rebounded 19.5% on 6.4% revenue growth, and net margin rose to 26.9%.
- Tesla's net income fell much faster than revenue in both years. In FY2024, revenue rose 0.9% while net income fell 52.7%, partly because FY2023 included a one-time tax benefit of roughly 5.9B and partly because operating income fell 20.4%. In FY2025, revenue fell 2.9% while net income fell 46.5%, as operating expenses rose 22.8% (mainly AI and R&D projects) and gross profit fell 2.0%. Net margin dropped from 15.5% to 4.0% over two years.

### Balance sheet
- Microsoft reduced its leverage. Liabilities grew more slowly than assets in both years (+18.4% vs. +24.3% in FY2024; +13.1% vs. +20.9% in FY2025), and the debt ratio fell from 49.9% to 44.5%. The FY2024 jump in assets reflects the Activision Blizzard acquisition, which closed in October 2023.
- Apple grew liabilities faster than assets in FY2024 (+6.1% vs. +3.5%), raising its debt ratio to 84.4%. Other current liabilities alone rose 19.5B, which lines up with the EU tax obligation. In FY2025, liabilities fell 7.3% and the debt ratio dropped to 79.5%.
- Apple's debt ratio of about 80% reflects capital returns rather than financial distress. In FY2025 it spent about 96.7B on buybacks and 15.4B on dividends, roughly equal to its 112.0B net income, which keeps shareholders' equity small relative to assets.
- Tesla's leverage is stable and the lowest of the three. Liabilities grew slightly faster than assets in FY2025 (+13.5% vs. +12.9%), but the debt ratio barely moved (39.6% to 39.9%).

### Cash generation
- Microsoft converts sales into cash most efficiently, with a FY2025 operating cash flow margin of 48.3% vs. 26.8% for Apple and 15.6% for Tesla. It generated more operating cash than Apple in FY2025 (136.2B vs. 111.5B) despite about 32% less revenue.
- Microsoft's and Apple's profits are backed by cash, with cash conversion at or above 1.0x every year (Microsoft 1.21x to 1.35x; Apple 1.00x to 1.26x).
- Tesla's FY2025 cash conversion of 3.89x is inflated by falling profit: operating cash flow held roughly flat (-1.2%) while net income fell 46.5%. Its FY2023 reading of 0.88x, the only one below 1.0x, reflects the non-cash tax benefit that boosted that year's net income.

### Limitations
- Fiscal years end in different months (Microsoft in June, Apple in late September, Tesla in December), so FY2025 covers a different 12-month period for each company.
- One-time items distort some year-over-year comparisons: Apple's FY2024 tax charge, Tesla's FY2023 tax benefit, and Apple's 53-week FY2023.
- Figures are nominal and not adjusted for inflation.
- Three annual data points show direction, not a statistically reliable trend.

### Implications for the GFC chatbot
- Ratios and growth rates are more useful than raw totals. Apple's revenue is 4.4x Tesla's, so dollar figures mostly reflect company size. Net margin, YoY growth, debt ratio, and cash conversion should be precomputed so the chatbot can answer them directly.
- Every value needs full labels: company, fiscal year, fiscal year-end date, units (USD millions), and source filing. Without the year-end date, "FY2025" is ambiguous across these companies.
- Numbers need context. "Tesla's net income fell 52.7% in FY2024" is accurate but misleading without the FY2023 tax benefit. The chatbot should attach known one-time items to affected figures and flag ratios like cash conversion when the denominator drops sharply.